# Регрессия для SI

Цель: предсказать индекс селективности SI по дескрипторам и сравнить несколько подходов регрессии с подбором гиперпараметров

In [21]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_csv('/Users/yaroslavbaev/Desktop/miphi/data/chem_data_prepared.csv')

X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['SI']

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [22]:
def reg_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred) 
    rmse = np.sqrt(mse)                       
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'RMSE': rmse,
        'R2': r2_score(y_true, y_pred),
    }

## Базовые линейные модели

In [23]:
ridge = Ridge(random_state=42)
ridge.fit(X_train_scaled, y_train)
ridge_val = reg_metrics(y_val, ridge.predict(X_val_scaled))

lasso = Lasso(random_state=42)
lasso.fit(X_train_scaled, y_train)
lasso_val = reg_metrics(y_val, lasso.predict(X_val_scaled))

ridge_val, lasso_val

({'mae': 96.06213488511797,
  'RMSE': np.float64(238.02970681730096),
  'R2': -13.075492965546125},
 {'mae': 73.61208857073295,
  'RMSE': np.float64(144.99092363834953),
  'R2': -4.222553812926606})

## RandomForestRegressor с подбором

In [4]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_params = {
    'n_estimators': [200, 400, 600],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
}

rf_search = RandomizedSearchCV(
    rf,
    rf_params,
    n_iter=25,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
rf_search.fit(X_train, y_train)
rf_search.best_params_, rf_search.best_score_

Fitting 3 folds for each of 25 candidates, totalling 75 fits


({'n_estimators': 600,
  'min_samples_split': 10,
  'min_samples_leaf': 4,
  'max_features': 'log2',
  'max_depth': 30},
 np.float64(-271.81344796471785))

In [5]:
rf_best = rf_search.best_estimator_
rf_val = reg_metrics(y_val, rf_best.predict(X_val))
rf_test = reg_metrics(y_test, rf_best.predict(X_test))
rf_val, rf_test

({'MAE': 43.084682320143976,
  'RMSE': np.float64(118.80720304485551),
  'R2': -2.5066048925215942},
 {'MAE': 195.6668267518164,
  'RMSE': np.float64(1355.8551260251515),
  'R2': 0.08480028425967401})

## XGBRegressor с подбором

In [6]:
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_estimators=500,
    n_jobs=-1,
)

xgb_params = {
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5, 7],
}

xgb_search = RandomizedSearchCV(
    xgb,
    xgb_params,
    n_iter=30,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
xgb_search.fit(X_train, y_train)
xgb_search.best_params_, xgb_search.best_score_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


({'subsample': 0.6,
  'min_child_weight': 7,
  'max_depth': 5,
  'learning_rate': 0.01,
  'colsample_bytree': 0.6},
 np.float64(-280.31997084233575))

In [7]:
xgb_best = xgb_search.best_estimator_
xgb_val = reg_metrics(y_val, xgb_best.predict(X_val))
xgb_test = reg_metrics(y_test, xgb_best.predict(X_test))
xgb_val, xgb_test

({'MAE': 41.37538005189755,
  'RMSE': np.float64(127.70051977826058),
  'R2': -3.0512273937820664},
 {'MAE': 195.97071279892575,
  'RMSE': np.float64(1347.1303025260486),
  'R2': 0.09654086707643761})

In [ ]:
# Логарифмирую таргет

In [8]:
df['SI_log'] = np.log1p(df['SI'])

In [9]:
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI', 'SI_log'])
y = df['SI_log']

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)


In [ ]:
# Повторим

In [18]:
rf_val, rf_test

({'MAE': 0.8200257516230842,
  'RMSE': np.float64(1.0480828253502756),
  'R2': 0.31679838203922983},
 {'MAE': 0.9085231918377473,
  'RMSE': np.float64(1.2084615226865747),
  'R2': 0.396539324540057})

In [17]:
xgb_val, xgb_test

({'MAE': 0.8164578068827419,
  'RMSE': np.float64(1.0793214540110772),
  'R2': 0.27546512140910906},
 {'MAE': 0.8813826019043134,
  'RMSE': np.float64(1.2012637606534542),
  'R2': 0.4037065049914923})

## Вывод

Изначально регрессия si в исходной шкале провалилась - модель ломалась на экстремальных значениях

С учётом того, что si — это отношение ic50 и cc50 и имеет тяжёлые хвосты, мы перешли к моделированию логарифма СИ. После этого качество резко улучшилось: для случайного леса на лог‑таргете удалось получить на валидации mae = 0.82, rmse = 1.05 и R2 = 0.32, а на тесте mae = 0.91, rmse = 1.21 и R2 = 0.40. Модель XGBRegressor показывает очень близкие результаты

Таким образом, в лог‑шкале SI модели уже способны объяснить заметную часть вариации индекса селективности и предсказывать, насколько сильно соединение отклоняется от среднего уровня селективности. В качестве рабочего решения можно использовать либо случайный лес, либо XGBRegressor, так как по качеству они практически равны на тестовой выборке